In [68]:
import os, csv
import pandas as pd

os.getcwd()

'/Users/caoziyi/Desktop/第三篇/MLLM-MSR-main/MLLM-MSR/data/microlens/preprocessing'

In [69]:
df = pd.read_csv('../../MicroLens-50k_pairs.csv')
print(f'shape: {df.shape}')
df[:5]
print(df['item'].nunique())

shape: (359708, 3)
19220


In [73]:
from collections import Counter
import numpy as np

min_u_num, min_i_num = 7, 5

def get_illegal_ids_by_inter_num(df, field, max_num=None, min_num=None):
    if field is None:
        return set()
    if max_num is None and min_num is None:
        return set()

    max_num = max_num or np.inf
    min_num = min_num or -1

    ids = df[field].values
    inter_num = Counter(ids)
    ids = {id_ for id_ in inter_num if inter_num[id_] < min_num or inter_num[id_] > max_num}
    print(f'{len(ids)} illegal_ids_by_inter_num, field={field}')

    return ids


def filter_by_k_core(df):
    while True:
        ban_users = get_illegal_ids_by_inter_num(df, field='user', max_num=None, min_num=min_u_num)
        ban_items = get_illegal_ids_by_inter_num(df, field='item', max_num=None, min_num=min_i_num)
        if len(ban_users) == 0 and len(ban_items) == 0:
            return

        dropped_inter = pd.Series(False, index=df.index)
        if 'user':
            dropped_inter |= df['user'].isin(ban_users)
        if 'item':
            dropped_inter |= df['item'].isin(ban_items)
        print(f'{len(dropped_inter)} dropped interactions')
        df.drop(df.index[dropped_inter], inplace=True)

In [74]:
filter_by_k_core(df)
print(f'k-core shape: {df.shape}')
print(f'shape after k-core: {df.shape}')
df[:5]

0 illegal_ids_by_inter_num, field=user
0 illegal_ids_by_inter_num, field=item
k-core shape: (145835, 3)
shape after k-core: (145835, 3)


,user,item,timestamp
0,36121,9580,1583378629552
2,37550,9580,1584412681021
3,14601,9580,1584848802432
4,15061,9580,1585388171106
6,3542,9580,1585404918503


In [75]:
print(df['user'].nunique())
print(df['item'].nunique())

14323
9535


In [76]:
import pandas as pd
import numpy as np

# df has three columns: 'user', 'item', 'timestamp'

# calculate the frequency of each item
item_frequency = df.groupby('item').size().reset_index(name='frequency')

# sort items by frequency in descending order
item_frequency_sorted = item_frequency.sort_values(by='frequency', ascending=False)

# sort the original dataframe by timestamp
df_sorted = df.sort_values('timestamp')

# get all unique items and users
all_items = set(df_sorted['item'].unique())
all_users = df_sorted['user'].unique()

# generate negative samples for each user
negative_samples_per_user = {}
for user in all_users:
    user_items = df_sorted[df_sorted['user'] == user]['item'].unique()
    available_items = list(all_items - set(user_items))
    negative_samples = np.random.choice(available_items, size=min(20, len(available_items)), replace=False)
    negative_samples_per_user[user] = negative_samples

print(len(negative_samples_per_user.keys()))

14323


In [21]:
top_6_items_per_user = {}
for user in all_users:
    # get the items interacted by the user
    user_items = df_sorted[df_sorted['user'] == user]['item']
    # get the top 6 items based on frequency
    top_6_items = user_items.map(item_frequency_sorted.set_index('item')['frequency']).sort_values(ascending=False).index[:6]
    # sorted these top 6 items by timestamp
    top_6_items_per_user[user] = df_sorted.loc[top_6_items].sort_value('timestamp')['item'].values

AttributeError: 'DataFrame' object has no attribute 'sort_value'

In [34]:
user = all_users[0]
user_items = df_sorted[df_sorted['user'] == user]['item']
top_6_items = (
        user_items
        .map(item_frequency_sorted.set_index('item')['frequency'])
        .sort_values(ascending=False)
        .index[:6]
    )
top_6_items_per_user[user] = df_sorted.loc[top_6_items].sort_values('timestamp')['item'].values


top_6_items_per_user[user]

array([16845, 15199,  9350, 17437, 12846,   675])

In [35]:
top_6_items_per_user = {}
for user in all_users:
    # 用户交互过的 items
    user_items = df_sorted[df_sorted['user'] == user]['item']

    # 找到这用户 top-6 高频 items
    top_6_items = (
        user_items
        .map(item_frequency_sorted.set_index('item')['frequency'])
        .sort_values(ascending=False)
        .index[:]
    )

    # 按 timestamp 排序
    top_6_items_per_user[user] = df_sorted.loc[top_6_items].sort_values('timestamp')['item'].values
top_6_items_per_user

{np.int64(36121): array([16845, 15199,  9350, 17437, 12846,   675]),
 np.int64(37550): array([16221,  8715,  8964,  2247, 19572, 14190]),
 np.int64(14601): array([14083, 12203, 18435,  7712,  4896, 18225]),
 np.int64(15061): array([ 7990, 12687, 13681, 19282, 10602, 10694]),
 np.int64(3542): array([ 3159, 18893,  6715, 19315,  9538, 16128]),
 np.int64(47592): array([14631,  9647, 17091,  3558, 12699, 17993]),
 np.int64(26119): array([14631, 15968,  2837,  7408,  8051, 19080]),
 np.int64(31970): array([ 7920,  5922, 13591, 17436,  6911,  3870]),
 np.int64(47771): array([14631, 19198,  4223,  3368, 15623, 13996]),
 np.int64(11254): array([14631, 15983, 14884, 16729, 17974,  8093]),
 np.int64(25902): array([ 7315, 15591, 14860,  6398, 16301, 17302]),
 np.int64(48195): array([14631, 17099,  7043, 12645,  3870,  7233]),
 np.int64(9851): array([ 7315, 15742,  5525,  6783,  8160, 15841]),
 np.int64(28036): array([14631,  7315,  1638, 11984,  2308,  7207]),
 np.int64(16718): array([14631,  216

In [62]:
item_frequency_sorted

,item,frequency
8168,16981,291
7574,15775,269
1560,3260,262
3141,6539,261
7947,16537,258
...,...,...
6071,12557,5
7446,15474,5
1025,2166,5
5600,11597,5


In [36]:
lines = []
for user in all_users:
    top_items_str = ', '.join([str(item) for item in top_6_items_per_user[user]])
    negative_samples_str = ', '.join([str(item) for item in negative_samples_per_user[user]])
    # format the line as "user_id\ttop_items\tnegative_samples"
    line = f"{user}\t{top_items_str}\t{negative_samples_str}"
    lines.append(line)

tsv_file_path = '../user_items_negs.tsv'
with open(tsv_file_path, 'w') as file:
    file.write('\n'.join(lines))

In [37]:
from sklearn.model_selection import train_test_split

tsv_file_path = '../user_items_negs.tsv'
data = pd.read_csv(tsv_file_path, sep='\t', header=None, names=['user', 'items', 'negative_samples'])

users = data['user'].unique()

train_users, test_val_users = train_test_split(users, test_size=0.2, random_state=42)

val_users, test_users = train_test_split(test_val_users, test_size=0.5, random_state=42)

train_data = data[data['user'].isin(train_users)]
val_data = data[data['user'].isin(val_users)]
test_data = data[data['user'].isin(test_users)]

print(f"Training data size: {len(train_data)}")
print(f"Validation data size: {len(val_data)}")
print(f"Test data size: {len(test_data)}")

Training data size: 11458
Validation data size: 1432
Test data size: 1433


In [38]:
def save_to_tsv(df, file_path):
    df.to_csv(file_path, sep='\t', header=False, index=False)
    print(f"File saved to {file_path}")

# save the dataframes to TSV files
save_to_tsv(train_data, '../train.tsv')
save_to_tsv(val_data, '../val.tsv')
save_to_tsv(test_data, '../test.tsv')


File saved to train.tsv
File saved to val.tsv
File saved to test.tsv
